In [2]:
from minsearch import Index
from gitsource import GithubRepositoryDataReader

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

# Q2

In [6]:
index= Index(
     text_fields=["content"],
     keyword_fields=["filename"]
     
)

index.fit(documents)

In [7]:
results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

In [8]:
print(results)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

# Q3 RAG

In [9]:
def build_context(search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            #lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()
context=build_context(results)
print(context)

01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call

In [10]:
INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

def build_prompt(query, search_results):
        context = build_context(search_results)
        return PROMPT_TEMPLATE.format(
            question=query, context=context
        )

query = "How does the agentic loop keep calling the model until it stops?"
prompt = build_prompt(query, results)
print(prompt)

QUESTION: How does the agentic loop keep calling the model until it stops?

CONTEXT:
01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the 

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from openai import OpenAI
import os

llm_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def llm(prompt, INSTRUCTIONS=INSTRUCTIONS):
        input_messages = [
            {'role': 'system', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]

        response = llm_client.chat.completions.create(
            model='poolside/laguna-m.1:free',
            messages=input_messages
        )

        return response.choices[0].message.content

output = llm(prompt)

In [13]:
output

'\nBased on the lesson content, the agentic loop keeps calling the model until it stops using a simple flag-based mechanism:\n\n**The key mechanism is the `has_function_calls` flag** that tracks whether the model\'s response contains any function calls. Here\'s how it works:\n\n1. **The loop runs indefinitely** (`while True:`) until explicitly broken\n\n2. **Each iteration checks the response** for function calls:\n   ```python\n   has_function_calls = False\n   \n   for item in response.output:\n       if item.type == "function_call":\n           # ... execute the function call\n           has_function_calls = True\n   ```\n\n3. **The loop only stops when there are no function calls** in the response:\n   ```python\n   if has_function_calls == False:\n       break\n   ```\n\nThe model decides when to stop by simply not returning any function calls in its response - that\'s the exit condition. As the lesson explains: "No function calls this turn means we\'re done. The loop stops when t

In [14]:
INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,

        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='poolside/laguna-m.1:free'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
           
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


In [15]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index,
    llm_client=llm_client,
)

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
     index,
     llm_client=llm_client,
)

def llm_with_token_limit(prompt):
     input_messages = [
          {'role': 'developer', 'content': assistant.instructions},
          {'role': 'user', 'content': prompt},
     ]
     return assistant.llm_client.responses.create(
          model=assistant.model,
          input=input_messages,
          max_output_tokens=1024,
     )

assistant.llm = llm_with_token_limit

answer = assistant.rag(query)
print(answer.output_text if hasattr(answer, "output_text") else answer)
print(answer)



Based on the lesson content, the agentic loop keeps calling the model until it stops through a simple flag-based mechanism:

**The Core Mechanism:**
The loop uses a `while True` construct that checks a `has_function_calls` flag after each model response. Here's how it works:

1. **Initial call**: The model receives the user's question along with the instructions and tools
2. **Check response**: After each call, the code examines whether the response contains any `function_call` entries
3. **Execute calls**: If function calls are present, the code executes them and appends the results to the message history
4. **Continue condition**: The `has_function_calls = True` flag is set whenever function calls are found
5. **Exit condition**: The loop only breaks (`break`) when `has_function_calls == False` - meaning the model returned a message with no tool calls

**Key Code Explained:**
```python
it = 1
while True:
    # ... make API call ...
    
    for item in response.output:
        if it

In [16]:
answer.usage.input_tokens

7552

# Q4

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

# Q5

In [18]:
# index chunk
def build_index(documents):
     index = Index(
          text_fields=["content"],
          keyword_fields=["filename"],
     )
     index.fit(documents)
     return index

index_chunk = build_index(chunks)

In [19]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index_chunk,
    llm_client=llm_client,
)

assistant.llm = llm_with_token_limit
answer = assistant.rag(query)
print(answer)


Response(id='gen-1782137280-34m3QmZICjqm9XAqgp3i', created_at=1782137280.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='poolside/laguna-m.1-20260312:free', object='response', output=[ResponseReasoningItem(id='rs_tmp_rcqwhyk45w8', summary=[], type='reasoning', content=[Content(text='\nThe user is asking how the agentic loop keeps calling the model until it stops. Let me look at the provided context to understand this.\n\nFrom the context, I can see the agentic loop code:\n\n```python\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model=model,\n        input=messages,\n        tools=[search_tool]\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n

In [20]:
answer.usage.input_tokens

2462

In [21]:
7114/2297

3.0970831519373094

# Q6

In [22]:
def search(query: str) -> str:
    """Search for information in the course lessons."""
    results = index_chunk.search(query, num_results=5)

    return "\n\n".join(
        doc["content"] for doc in results
    )

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [24]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search for information in the course lessons.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client,
)


In [26]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)

print(result)
print("Search calls:", search_counter)

AttributeError: 'OpenAI' object has no attribute 'send_request'

In [27]:
search_counter = 0

def search(query):
    global search_counter
    search_counter += 1
    return index.search(query=query, num_results=5)

In [28]:
def build_context(results):
    context = ""

    for r in results:
        context += f"""
Filename: {r['filename']}
Content:
{r['content']}
"""
    return context

In [31]:
def agent(question):
    queries = [
        question,
        "agentic loop",
        "plain RAG",
        "how agent calls model repeatedly"
    ]

    all_results = []

    for q in queries:
        results = search(q)
        all_results.extend(results)

    context = build_context(all_results)

    prompt = f"""
You're a course teaching assistant.

Answer using the context below.

Question:
{question}

Context:
{context}
"""

    response = llm_client.responses.create(
        model="poolside/laguna-m.1:free",
        input=prompt
    )

    return response.output_text

In [32]:
search_counter = 0

answer = agent(
    "How does the agentic loop work, and how is it different from plain RAG?"
)

print(answer)
print("Search calls:", search_counter)


## How the Agentic Loop Works, and How It Differs From Plain RAG

### The Agentic Loop

Based on the context provided, the **agentic loop** is a dynamic, iterative process where an LLM controls the flow of execution to solve a task. Here's how it works:

1.  **Control Flow**: Unlike a fixed pipeline, the LLM is placed in the driver's seat. It determines which actions to take and in what order.
2.  **Core Components**: An agent consists of three main parts wired together in a loop:
    *   **Instructions (Developer Prompt):** Defines the agent's role, behavior, and goals. The quality of these instructions heavily influences the agent's helpfulness.
    *   **Tools:** Functions the agent can call to perform actions (e.g., `search`). The LLM decides when and with what arguments to call these tools based on its reasoning.
    *   **Memory (Message History):** The agent maintains a record of the entire conversation (`messages` list). This includes the initial prompt, every response from th